# Train YOLO Object Detection Models

This notebook provides a complete pipeline for training YOLO models on custom datasets:
1. Download dataset from a user-provided URL
2. Extract and prepare the data
3. Split into training/validation sets
4. Configure YOLO training parameters
5. Train the YOLO model
6. Download the trained model weights

**Note**: This notebook is designed to run on Google Colab or other cloud environments with GPU support.

In [ ]:
#!/usr/bin/env python3
"""
Complete YOLO Model Training Pipeline in One Step
Downloads dataset, prepares it, configures, and trains a YOLO model
"""

from IPython.display import display, FileLink, HTML, update_display
from ipywidgets import Text, Button, Output, VBox, Label
import urllib.parse
import subprocess
import os
import shutil
import zipfile
import yaml

# ==============================================================================
# STEP 1: USER INPUT INTERFACE FOR TOKEN
# ==============================================================================
output = Output()
token_input = Text(
    placeholder='Paste your dataset download URL here',
    description='Dataset URL:',
    style={'description_width': '100px'},
    layout={'width': '80%'}
)
status_label = Label(value="Status: Waiting for input...")
download_button = Button(description='Start Complete Training Pipeline', button_style='success')
dataset_url = None

def validate_url(url):
    """Validate URL format"""
    try:
        result = urllib.parse.urlparse(url.strip())
        return all([result.scheme, result.netloc])
    except:
        return False

def on_start_click(b):
    global dataset_url
    with output:
        output.clear_output()
        dataset_url = token_input.value.strip()
        
        if not dataset_url:
            print("❌ Error: Please enter a dataset URL")
            status_label.value = "Status: No URL provided"
            return
        
        if not validate_url(dataset_url):
            print("❌ Error: Invalid URL format")
            status_label.value = "Status: Invalid URL format"
            return
        
        print("✅ URL validated. Starting training pipeline...\n")
        status_label.value = "Status: Pipeline running..."
        run_complete_pipeline()

download_button.on_click(on_start_click)

display(VBox([
    HTML("<h2>🚀 YOLO Model Training - Complete Pipeline</h2>"),
    HTML("<p>Enter your dataset download URL and click Start to begin the entire training process in one go.</p>"),
    token_input,
    download_button,
    status_label,
    output
]))

# ==============================================================================
# COMPLETE PIPELINE FUNCTION
# ==============================================================================
def run_complete_pipeline():
    """Execute the complete training pipeline"""
    try:
        # Setup directories
        os.makedirs('/content', exist_ok=True)
        os.makedirs('/content/custom_data', exist_ok=True)
        
        # STEP 2: DOWNLOAD DATASET
        print("\n" + "="*70)
        print("STEP 2: DOWNLOADING DATASET")
        print("="*70)
        print(f"📥 Downloading from: {dataset_url}")
        
        result = subprocess.run(
            ['wget', '-O', '/content/data.zip', dataset_url],
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if result.returncode != 0:
            print(f"❌ Download failed: {result.stderr}")
            status_label.value = "Status: Download failed"
            return
        
        file_size = os.path.getsize('/content/data.zip') / (1024**2)
        print(f"✅ Download completed! ({file_size:.2f} MB)\n")
        
        # STEP 3: EXTRACT DATASET
        print("="*70)
        print("STEP 3: EXTRACTING DATASET")
        print("="*70)
        print("📂 Extracting to /content/custom_data...")
        
        with zipfile.ZipFile('/content/data.zip', 'r') as zip_ref:
            zip_ref.extractall('/content/custom_data')
        
        print("✅ Extraction completed!\n")
        
        # STEP 4: SPLIT DATA
        print("="*70)
        print("STEP 4: SPLITTING DATA INTO TRAIN/VALIDATION")
        print("="*70)
        
        # Download split script
        print("📥 Downloading train_val_split.py script...")
        result = subprocess.run(
            ['wget', '-O', '/content/train_val_split.py',
             'https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py'],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        # Run split script
        print("🔄 Splitting dataset (90% train / 10% validation)...")
        result = subprocess.run(
            ['python', '/content/train_val_split.py',
             '--datapath=/content/custom_data',
             '--train_pct=0.9'],
            capture_output=True,
            text=True,
            timeout=120
        )
        print(result.stdout)
        print("✅ Data split completed!\n")
        
        # STEP 5: INSTALL LIBRARIES
        print("="*70)
        print("STEP 5: INSTALLING REQUIRED LIBRARIES")
        print("="*70)
        print("📦 Installing ultralytics...")
        
        result = subprocess.run(
            [subprocess.sys.executable, '-m', 'pip', 'install', 'ultralytics', '-q'],
            capture_output=True,
            text=True,
            timeout=300
        )
        print("✅ Installation completed!\n")
        
        # STEP 6: CREATE DATA.YAML CONFIG
        print("="*70)
        print("STEP 6: CREATING DATA.YAML CONFIGURATION")
        print("="*70)
        
        path_to_classes_txt = '/content/custom_data/classes.txt'
        path_to_data_yaml = '/content/data.yaml'
        
        if not os.path.exists(path_to_classes_txt):
            print(f"❌ classes.txt not found at {path_to_classes_txt}")
            status_label.value = "Status: Missing classes.txt"
            return
        
        with open(path_to_classes_txt, 'r') as f:
            classes = [line.strip() for line in f.readlines() if line.strip()]
        
        print(f"✅ Found {len(classes)} classes: {', '.join(classes)}\n")
        
        data = {
            'path': '/content/custom_data',
            'train': 'train/images',
            'val': 'validation/images',
            'nc': len(classes),
            'names': classes
        }
        
        with open(path_to_data_yaml, 'w') as f:
            yaml.dump(data, f, sort_keys=False)
        
        print(f"✅ Created data.yaml:\n")
        with open(path_to_data_yaml, 'r') as f:
            print(f.read())
        
        # STEP 7: TRAIN MODEL
        print("="*70)
        print("STEP 7: TRAINING YOLO MODEL")
        print("="*70)
        print("🚀 Starting YOLO11s training (60 epochs, 640x640)...")
        print("⏱️  This may take 30min - several hours depending on GPU and dataset size\n")
        
        result = subprocess.run(
            ['yolo', 'detect', 'train',
             'data=/content/data.yaml',
             'model=yolo11s.pt',
             'epochs=60',
             'imgsz=640',
             'patience=20',
             'device=0'],
            timeout=None
        )
        
        print("\n✅ Training completed!\n")
        
        # STEP 8: PREPARE MODEL FOR DOWNLOAD
        print("="*70)
        print("STEP 8: DOWNLOADING TRAINED MODEL")
        print("="*70)
        
        model_source = '/content/runs/detect/train/weights/best.pt'
        model_destination = '/content/best_model.pt'
        
        if not os.path.exists(model_source):
            print(f"❌ Trained model not found at {model_source}")
            status_label.value = "Status: Model not found"
            return
        
        file_size_mb = os.path.getsize(model_source) / (1024 * 1024)
        print(f"✅ Model ready! Size: {file_size_mb:.2f} MB\n")
        
        shutil.copy2(model_source, model_destination)
        
        print("📥 Your trained model is ready for download:")
        display(FileLink(model_destination, result_html_prefix="📥 Click to download: "))
        
        print(f"\n✅ Complete pipeline finished successfully!")
        print(f"📁 Results location: /content/runs/detect/train/")
        print(f"📦 Model file: {model_destination}")
        
        status_label.value = "Status: Pipeline completed successfully! ✅"
        
    except KeyboardInterrupt:
        print("\n⚠️  Training interrupted by user")
        status_label.value = "Status: Interrupted by user"
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        status_label.value = f"Status: Error - {str(e)}"
        import traceback
        traceback.print_exc()